In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
# Load data into NumPy arrays
data = load_iris()
X, y = data["data"], data["target"]
# convert NumPy array into PyTorch tensors
X = torch.tensor(X, dtype=torch.float32)
y = torch.tensor(y, dtype=torch.long)
# split
X_train, X_test, y_train, y_test = train_test_split(X, y, train_size=0.7, shuffle=True)
# PyTorch model
class Multiclass(nn.Module):
    def __init__(self):
        super().__init__()
        self.hidden = nn.Linear(4, 8)
        self.act = nn.ReLU()
        self.output = nn.Linear(8, 3)
        self.logsoftmax = nn.LogSoftmax(dim=1)

    def forward(self, x):
        x = self.act(self.hidden(x))
        x = self.logsoftmax(self.output(x))
        return x
model = Multiclass()
# loss metric and optimizer
loss_fn = nn.NLLLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)
# prepare model and training parameters
n_epochs = 100
batch_size = 5
batch_start = torch.arange(0, len(X), batch_size)
# training loop
for epoch in range(n_epochs):
    for start in batch_start:
# take a batch
        X_batch = X_train[start:start+batch_size]
        y_batch = y_train[start:start+batch_size]
# forward pass
        y_pred = model(X_batch)
        loss = loss_fn(y_pred, y_batch)
# backward pass
        optimizer.zero_grad()
        loss.backward()
# update weights
        optimizer.step()
y_pred = model(X_test)
acc = (torch.argmax(y_pred, 1) == y_test).float().mean()
print("Accuracy: %.2f" % acc)

Accuracy: 0.96


In [2]:
optimizer = optim.Adam(model.parameters(), lr=0.001)
# create a new model
newmodel = Multiclass()
# ask PyTorch to ignore autograd on update and overwrite parameters
with torch.no_grad():
    for newtensor, oldtensor in zip(newmodel.parameters(), model.parameters()):
        newtensor.copy_(oldtensor)
# test with new model using copied tensor
y_pred = newmodel(X_test)
acc = (torch.argmax(y_pred, 1) == y_test).float().mean()
print("Accuracy: %.2f" % acc)

Accuracy: 0.96


In [3]:
import pprint
pp = pprint.PrettyPrinter(indent=4)
pp.pprint(model.state_dict())

OrderedDict([   (   'hidden.weight',
                    tensor([[ 0.6272,  1.0845, -0.1345, -0.5963],
        [ 0.1427, -0.4118, -0.1367,  0.1652],
        [-0.1552, -0.0506,  0.2679,  1.0476],
        [-0.4524, -0.0744, -0.1365,  0.1978],
        [-0.1757, -0.0150,  0.2496,  0.5168],
        [-0.1132, -0.2870,  0.7330,  1.3971],
        [-0.0027, -0.2763,  0.7367,  1.2409],
        [-0.3903, -0.2025, -0.3245, -0.0776]])),
                (   'hidden.bias',
                    tensor([ 0.3973, -0.2665, -0.7564, -0.4501, -0.5920, -0.0752, -0.4310,  0.4646])),
                (   'output.weight',
                    tensor([[ 0.5315,  0.2324, -0.9992, -0.2755, -0.0830, -1.1460, -1.0270,  0.0293],
        [-0.0684,  0.1823, -0.4711,  0.0939, -0.5413, -0.1610,  0.0689,  0.2712],
        [-0.6890, -0.1852,  0.5892,  0.1606,  0.2301,  0.2915,  0.3188,  0.2442]])),
                ('output.bias', tensor([ 0.3837,  0.4219, -0.3749]))])


In [4]:
import pickle
# Save model
with open("iris-model.pickle", "wb") as fp:
    pickle.dump(model.state_dict(), fp)
# Create new model and load states
newmodel = Multiclass()
with open("iris-model.pickle", "rb") as fp:
    newmodel.load_state_dict(pickle.load(fp))
# test with new model using copied tensor
y_pred = newmodel(X_test)
acc = (torch.argmax(y_pred, 1) == y_test).float().mean()
print("Accuracy: %.2f" % acc)


Accuracy: 0.96


In [5]:
# Save model
torch.save(model.state_dict(), "iris-model.pth")
# Create new model and load states
newmodel = Multiclass()
newmodel.load_state_dict(torch.load("iris-model.pth"))
# test with new model using copied tensor
y_pred = newmodel(X_test)
acc = (torch.argmax(y_pred, 1) == y_test).float().mean()
print("Accuracy: %.2f" % acc)   

Accuracy: 0.96


In [7]:
# Save model
torch.save(model, "iris-model-full.pth")
# Load model
newmodel = torch.load("iris-model-full.pth")
# test with new model using copied tensor
y_pred = newmodel(X_test)
acc = (torch.argmax(y_pred, 1) == y_test).float().mean()
print("Accuracy: %.2f" % acc)

UnpicklingError: Weights only load failed. This file can still be loaded, to do so you have two options, [1mdo those steps only if you trust the source of the checkpoint[0m. 
	(1) In PyTorch 2.6, we changed the default value of the `weights_only` argument in `torch.load` from `False` to `True`. Re-running `torch.load` with `weights_only` set to `False` will likely succeed, but it can result in arbitrary code execution. Do it only if you got the file from a trusted source.
	(2) Alternatively, to load with `weights_only=True` please check the recommended steps in the following error message.
	WeightsUnpickler error: Unsupported global: GLOBAL __main__.Multiclass was not an allowed global by default. Please use `torch.serialization.add_safe_globals([__main__.Multiclass])` or the `torch.serialization.safe_globals([__main__.Multiclass])` context manager to allowlist this global if you trust this class/function.

Check the documentation of torch.load to learn more about types accepted by d

In [8]:
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
# Load data into NumPy arrays
data = load_iris()
X, y = data["data"], data["target"]
# convert NumPy array into PyTorch tensors
X = torch.tensor(X, dtype=torch.float32)
y = torch.tensor(y, dtype=torch.long)
# split
X_train, X_test, y_train, y_test = train_test_split(X, y, train_size=0.7, shuffle=True)
# PyTorch model
class Multiclass(nn.Module):
    def __init__(self):
        super().__init__()
        self.hidden = nn.Linear(4, 8)
        self.act = nn.ReLU()
        self.output = nn.Linear(8, 3)
        self.logsoftmax = nn.LogSoftmax(dim=1)
    def forward(self, x):
        x = self.act(self.hidden(x))
        x = self.logsoftmax(self.output(x))
        return x

model = Multiclass()
# loss metric and optimizer
loss_fn = nn.NLLLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)
# prepare model and training parameters
n_epochs = 100
batch_size = 5
batch_start = torch.arange(0, len(X), batch_size)
# training loop
for epoch in range(n_epochs):
    for start in batch_start:
# take a batch
        X_batch = X_train[start:start+batch_size]
        y_batch = y_train[start:start+batch_size]
# forward pass
        y_pred = model(X_batch)
        loss = loss_fn(y_pred, y_batch)
# backward pass
        optimizer.zero_grad()
        loss.backward()
# update weights
        optimizer.step()
# Save model
torch.save(model.state_dict(), "iris-model.pth")

In [9]:
import torch
import torch.nn as nn
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
# Load data into NumPy arrays
data = load_iris()
X, y = data["data"], data["target"]
# convert NumPy array into PyTorch tensors
X = torch.tensor(X, dtype=torch.float32)
y = torch.tensor(y, dtype=torch.long)
# split
X_train, X_test, y_train, y_test = train_test_split(X, y, train_size=0.7, shuffle=True)
# PyTorch model
class Multiclass(nn.Module):
    def __init__(self):
        super().__init__()

        self.hidden = nn.Linear(4, 8)
        self.act = nn.ReLU()
        self.output = nn.Linear(8, 3)
        self.logsoftmax = nn.LogSoftmax(dim=1)
    def forward(self, x):
        x = self.act(self.hidden(x))
        x = self.logsoftmax(self.output(x))
        return x
# Create new model and load states
model = Multiclass()
model.load_state_dict(torch.load("iris-model.pth"))
# Run model for inference
y_pred = model(X_test)
acc = (torch.argmax(y_pred, 1) == y_test).float().mean()
print("Accuracy: %.2f" % acc)

Accuracy: 0.96
